# 04 — Reconstruction Improvement

**Question:** Does pre-filtering or event-aware compression improve transient recovery?

In [ ]:
import sys
sys.path.insert(0, '../..')
import numpy as np
from src.data_loader.loader import load_signal
from src.generate_synthetic import create_synthetic_dataset, get_phase_segments
from src.preprocessing.noise import apply_all_noise
from src.v2.denoising.pipeline import denoise_multistage
from src.compression.fft_compression import compress_fft, decompress_fft
from src.v2.compression.hybrid import compress_hybrid, decompress_hybrid
from src.v2.compression.event_aware import compress_event_aware, decompress_event_aware
from src.metrics.snr import snr_db
from src.metrics.segmented import compute_segment_metrics
from src.visualization.error_plots import plot_error_analysis
%matplotlib inline

In [ ]:
path = create_synthetic_dataset()
time, clean, meta = load_signal(path)
fs = meta['sampling_frequency']
noisy, _ = apply_all_noise(clean, seed=42)
working, _ = denoise_multistage(noisy, fs)
methods = {
    'FFT 10% (no pre-denoise)': (compress_fft(noisy, 0.1), decompress_fft),
    'Hybrid (denoised)': (compress_hybrid(working, 0.85), decompress_hybrid),
    'Event-aware (denoised)': (compress_event_aware(working, fs), decompress_event_aware),
}
for name, (comp, decomp) in methods.items():
    recon = decomp(comp)
    segs = compute_segment_metrics(time, clean, recon, get_phase_segments())
    launch_snr = segs.get('launch', {}).get('snr_db', 'N/A')
    print(f'{name:30s}  overall SNR vs clean={snr_db(clean, recon):.1f} dB  launch SNR={launch_snr} dB')